### Tools
##### A tool is a function that an LLM can use to perform a specific task.

In [1]:
from langchain.chat_models import init_chat_model
import os
from dotenv import load_dotenv
from langchain.tools import tool
load_dotenv()

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")


In [2]:
# we can also bind the tools like direct with the agent 
@tool
def get_whether(location:str) -> str:
    """ this function gives the whether information """
    print("localtion",location)
    return f"the current wether of this {location} is sunny"

In [3]:
models=init_chat_model(model="gemini-2.5-flash", model_provider="google_genai")
models_bind_tools=models.bind_tools([get_whether]) 
response=models_bind_tools.invoke("what is ahmedabads wether")
print(response)

content='' additional_kwargs={'function_call': {'name': 'get_whether', 'arguments': '{"location": "Ahmedabad"}'}, '__gemini_function_call_thought_signatures__': {'2498c1fa-8d2c-468f-a609-915c0a446bd9': 'CpQCARFNMg+h6/X+6N6M0twRZDrX/HkhjLHe7DzDYWbYEhJBCq3N0MVnsX25m16SYjcU373lyf/Zmh+6QKNR0FSFgkUjZWANCsHu7s5iT1Iuqdlvdq7tvPtu7wYJ63juRtkwMl4hxhDLP1AYyi5MF8XH+pFceyD8bvF+IhRArXTK59TpG88+Ill3IMDRElEf8zl82FLrKoAmzHrmaXRr1kJZ0be7Z0MYZES+/EWTymSOCIh+5tPeZO2IxcIriIOEFS3fGyVwMljpIToBDARAeE+pmTEFmvmDh9FTonekmkvlG+pwRwZUYr4ZDTOo/QSZMnJwh7XixFi/jZqAr/QotJ9fWTZ4sHNN+NqfgWyl3Cn2/eP8/aZc'}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a055f5-99bc-76d2-8eb8-9e8e4dffa196-0' tool_calls=[{'name': 'get_whether', 'args': {'location': 'Ahmedabad'}, 'id': '2498c1fa-8d2c-468f-a609-915c0a446bd9', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 47, 'output_tokens': 75, 'total_tokens': 1

In [10]:
print(response.tool_calls[0])
for tool_call in response.tool_calls:
    print(tool_call["name"])
    print(tool_call["args"])

{'name': 'get_whether', 'args': {'location': 'Ahmedabad'}, 'id': '2498c1fa-8d2c-468f-a609-915c0a446bd9', 'type': 'tool_call'}
get_whether
{'location': 'Ahmedabad'}


#### Tools Execution Loop

In [5]:
messages = [{"role": "user", "content": "What's the weather in Boston?"}] 
ai_msg = models_bind_tools.invoke(messages) 
print("0",ai_msg)
print("0.1",ai_msg.tool_calls)

messages.append(ai_msg) 
print("1",messages)


0 content='' additional_kwargs={'function_call': {'name': 'get_whether', 'arguments': '{"location": "Boston"}'}, '__gemini_function_call_thought_signatures__': {'1ea567ee-d928-4879-97e1-b555660ffa6f': 'Ct0BARFNMg+6MppD0mrhPUlQ09flHmFyM0HeVEXriRhLvmgh5pLghEIHqELCXwfD41j/H0IrOEJTm4c4LPS3tvtph/4AFdQApMcpOIhtaa/iaJXRQfWt07zW2zsMr0mskxWGEHOxXIvKDUo7ZpuUMi0Wk6KPoKasySuqgvc0bv0dqIrn2CZfDoc2dqnzCwluBFqh9oyGtorBbCrIRAoSKd1yGqXCaCwvnqjhl5cQXcirn+qVvO1JZEYxLyvMIvzV+OiKg8m2EEaMxuuz4w8GPejWU+5ihjCluW4/ddgLzB0='}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a055f5-a1f1-77f1-8552-f9e4b4fe06f0-0' tool_calls=[{'name': 'get_whether', 'args': {'location': 'Boston'}, 'id': '1ea567ee-d928-4879-97e1-b555660ffa6f', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 47, 'output_tokens': 58, 'total_tokens': 105, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reas

In [6]:
# Step 2: Execute tools and collect results  
# 0.1 [{'name': 'get_whether', 'args': {'location': 'Boston'}, 'id': '77bdcd83-1bcf-4dd1-9e1d-d173de05849d', 'type': 'tool_call'}]
for tool_call in ai_msg.tool_calls: 
    print(tool_call)
    tool_result = get_whether.invoke(tool_call) 
    messages.append(tool_result) 
    
# Step 3: Pass results back to model for final response 

final_response = models_bind_tools.invoke(messages) 
print(final_response.text)

{'name': 'get_whether', 'args': {'location': 'Boston'}, 'id': '1ea567ee-d928-4879-97e1-b555660ffa6f', 'type': 'tool_call'}
localtion Boston
The current weather in Boston is sunny.
